# 05 — Error Analysis and Final Model

Picks up from notebook 03's XGBoost model (AUC-ROC 0.751, AUC-PR 0.532,
cost ₹641,050 at threshold 0.51). Precision (0.37) was better than the
logistic regression baseline but still modest, so this notebook digs into
*where* the model is wrong, rather than continuing to tune blindly.

Assumes `model_xgb`, `probs_xgb`, `x_train`, `x_test`, `y_train`, `y_test`,
`num_features_v2`, `cat_features` from notebook 03 are available (either by
re-running 03 in the same session, or by re-executing its cells here).


## KS statistic

Standard credit-risk-style separation metric — how far apart the model pushes the true-positive and false-positive rates across thresholds.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np

fpr, tpr, roc_thresholds = roc_curve(y_test, probs_xgb)
ks_stat = max(tpr - fpr)
ks_threshold = roc_thresholds[np.argmax(tpr - fpr)]
print(f"KS Statistic: {ks_stat:.4f} at threshold {ks_threshold:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(roc_thresholds, tpr, label="True Positive Rate - Bad Orders Caught")
plt.plot(roc_thresholds, fpr, label="False Positive Rate - Good Orders Flagged")
plt.axvline(ks_threshold, color="gray", linestyle="--", label=f"Max separation at {ks_threshold:.2f}")
plt.gca().invert_xaxis()
plt.xlabel("Threshold")
plt.ylabel("Rate")
plt.title(f"KS Statistic = {ks_stat:.3f}")
plt.legend()
plt.tight_layout()
plt.savefig("../docs/ks_curve.png", dpi=150)
plt.show()


## Precision-recall curve

The chosen-threshold marker is read directly from the `precision_recall_curve` output arrays, not hardcoded, so it can't drift out of sync with whatever the model actually produces.

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, probs_xgb)
baseline = y_test.mean()

chosen_threshold = 0.51
idx = np.argmin(np.abs(pr_thresholds - chosen_threshold))
chosen_precision = precision[idx]
chosen_recall = recall[idx]
print(f"At threshold {pr_thresholds[idx]:.4f}: precision={chosen_precision:.3f}, recall={chosen_recall:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label="Model")
plt.axhline(baseline, color="gray", linestyle="--", label=f"Baseline (positive rate = {baseline:.2f})")
plt.scatter([chosen_recall], [chosen_precision], color="red", zorder=5,
            label=f"Chosen Threshold ({pr_thresholds[idx]:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig("../docs/pr_curve.png", dpi=150)
plt.show()


## Error analysis: where are the false positives concentrated?

Rather than eyeballing individual wrong predictions, we check whether any
segment shows up in false positives *more than its overall share of the
data would predict* — a `fp_over_index` well above 1 signals real
concentration, not noise.


In [ ]:
threshold = 0.5099999999999998

results = x_test[num_features_v2 + cat_features].copy()
results["true_label"] = y_test.values
results["predicted_prob"] = probs_xgb
results["predicted_label"] = (probs_xgb > threshold).astype(int)

def outcome_type(row):
    if row["predicted_label"] == 1 and row["true_label"] == 0:
        return "False Positive"
    elif row["predicted_label"] == 0 and row["true_label"] == 1:
        return "False Negative"
    elif row["predicted_label"] == 1 and row["true_label"] == 1:
        return "True Positive"
    else:
        return "True Negative"

results["outcome"] = results.apply(outcome_type, axis=1)
false_positives = results[results["outcome"] == "False Positive"]
print(f"Total false positives: {len(false_positives)} out of {len(results)} test orders")

def concentration_check(df, fp_df, column, min_count=30):
    overall_dist = df[column].value_counts(normalize=True)
    fp_dist = fp_df[column].value_counts(normalize=True)
    overall_counts = df[column].value_counts()
    comparison = pd.DataFrame({
        "overall_share": overall_dist, "fp_share": fp_dist, "total_count": overall_counts,
    }).fillna(0)
    comparison["fp_over_index"] = comparison["fp_share"] / comparison["overall_share"]
    return comparison[comparison["total_count"] >= min_count].sort_values("fp_over_index", ascending=False)

import pandas as pd
print("Product Category"); print(concentration_check(results, false_positives, "product_category_name").head(10))
print("\nCustomer Region"); print(concentration_check(results, false_positives, "customer_state").head(10))
print("\nPayment Type"); print(concentration_check(results, false_positives, "payment_type").head(10))

def numeric_concentration_check(df, fp_df, column, bins=5):
    df_binned = df.copy()
    df_binned[f"{column}_bin"] = pd.qcut(df_binned[column], q=bins, duplicates="drop")
    fp_binned = df_binned.loc[fp_df.index]
    return concentration_check(df_binned, fp_binned, f"{column}_bin", min_count=20)

for col in ["price", "freight_ratio", "seller_order_count", "seller_bad_rate", "delay_days"]:
    print(f"\n=== {col} ===")
    print(numeric_concentration_check(results, false_positives, col))


**Finding**: `seller_bad_rate`'s top bin (0.196–0.472) showed `fp_over_index ≈ 3.4` —
nearly 70% of all false positives came from orders where the seller's
(shrunk) historical bad rate was already elevated. `Office Furniture`
(4.24x) was a secondary, smaller pattern. This warranted a closer look:
is the model overreacting to seller reputation, or is that segment
genuinely riskier?


## Confirming with SHAP feature importance

In [ ]:
explainer_check = shap.TreeExplainer(model_xgb.named_steps["clf"])
x_test_transformed = model_xgb.named_steps["prep"].transform(x_test[num_features_v2 + cat_features])
if hasattr(x_test_transformed, "toarray"):
    x_test_transformed = x_test_transformed.toarray()
shap_values_all = explainer_check.shap_values(x_test_transformed)

mean_abs_shap = np.abs(shap_values_all).mean(axis=0)
feature_names = model_xgb.named_steps["prep"].get_feature_names_out()
importance_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap}).sort_values("mean_abs_shap", ascending=False)
print(importance_df.head(15))


`delay_days` (0.506) and `seller_bad_rate` (0.488) were close at the top — not one feature dominating outright, which ruled out the simplest explanation.

## Is the flag rate proportionate to the actual risk in that segment?

In [ ]:
top_bin_mask = results["seller_bad_rate"] > 0.196

print("Actual bad-order rate within this bin:", results.loc[top_bin_mask, "true_label"].mean())
print("Actual bad-order rate overall:", results["true_label"].mean())
print("Flag rate within this bin:", results.loc[top_bin_mask, "predicted_label"].mean())
print("Flag rate overall:", results["predicted_label"].mean())
print("\nPredicted probability spread within this bin:")
print(results.loc[top_bin_mask, "predicted_prob"].describe())


**Result**: actual bad-rate in this bin was 24.2% (vs. 15.3% overall — a real,
1.6x elevation), but the flag rate was 69.5% — far more aggressive than the
underlying risk alone would suggest. The probability spread (std 0.145,
range 0.25–0.999) showed the model still differentiates *within* the bin —
this wasn't a "can't tell orders apart" problem, the whole distribution was
just shifted too far right.

## Mitigations attempted

Five different fixes were tried, tested via the same bin-level check each
time (`actual bad-rate` vs. `flag rate` within the top `seller_bad_rate` bin,
evaluated at *each version's own* cost-optimal threshold, not a fixed 0.5 —
an early version of this comparison was misleading exactly because it
compared different models at the same threshold rather than each at its own
optimum):

| Attempt | AUC-ROC | AUC-PR | Cost | Flag rate in top bin |
|---|---|---|---|---|
| Baseline (this notebook's model) | 0.751 | 0.532 | ₹641,050 | 69.5% |
| Shrinkage weight 20 → 50 | — | — | — | 74.5% (worse) |
| `max_depth=3`, `reg_lambda=5` | 0.747 | 0.527 | — | 71.9% |
| + `monotone_constraints` on seller_bad_rate | 0.747 | 0.528 | ₹647,650 | 75.5% |
| + `colsample_bytree/bynode=0.7`, `min_child_weight=10`, tempered `scale_pos_weight` | 0.746 | 0.527 | ₹643,450 (at threshold 0.5) | 36.0% (at threshold 0.5) |
| — same, re-evaluated at its own optimal threshold (0.39) | — | — | ₹643,450 | **71.4%** (regression re-appears) |

The last row matters most: the apparent fix in the second-to-last row was an
artifact of comparing at the wrong threshold. Tempering `scale_pos_weight`
shifted the whole probability distribution downward, which moved the
cost-optimal threshold from ~0.5 to 0.39 — once re-evaluated at *its own*
correct threshold, the same ~70% flag rate re-emerged. None of the five
architectural interventions changed the underlying behavior; they only
appeared to, until evaluated consistently.

## Conclusion

Every variant converges to flagging roughly 70% of orders from this segment.
Given the cost asymmetry (a missed bad order costs 6x a false flag), and
a 1.6x elevated true bad-rate in this segment, this level of aggressive
flagging is plausibly the model correctly executing the stated cost
function in the one segment where the base rate makes it worthwhile — not
an instability or overfitting artifact. This is treated as a documented,
defensible characteristic of the model rather than a bug, and the simpler
unconstrained model (no `reg_lambda`, no `monotone_constraints`, no
`colsample`/`min_child_weight` overrides, full `scale_pos_weight`) is kept
as the final model, since none of the added complexity improved on it.


## Revisiting the cost assumption

Everything above (the mitigation attempts, the seller_bad_rate over-flagging
finding) was evaluated under one assumption: a missed bad order costs 6x
a false flag (`cost_fp=50`, `cost_fn=300`). That assumption was never
actually validated — it was a starting guess. Worth checking against real
industry figures before treating threshold 0.51 as "the" answer.


**Industry research on ecommerce fraud/risk detection consistently finds
the opposite of what was assumed**: false positives (wrongly flagging a
good order/customer) tend to cost merchants *more* than the fraud or bad
orders they prevent, not less:

- Merchants lose an estimated 2.79% of revenue to false positives vs. 0.52%
  to actual chargebacks/fraud — roughly a 5:1 ratio favoring FP as costlier
- Industry-wide, ecommerce false-positive losses (~$443B) vastly exceed
  actual fraud losses (~$6.4B)
- A frequently cited figure: $13 lost to false positives for every $1 of
  genuine fraud

Olist itself has no direct cost fields (no chargeback amounts, no customer
lifetime value, no refund-processing cost) to calibrate against directly —
the best available proxy is average order value, which sets the scale of
what's plausible but doesn't pin down an exact ratio.


## Three cost scenarios, not one

Rather than replace 6:1 with a single new number, three scenarios are
compared, since the "correct" ratio is a business judgment call this
project can't fully resolve with the data available:


In [ ]:
cost_scenarios_to_test = {
    "6:1 (FN costly, original assumption)": (50, 300),
    "3:1 (FP costly, milder)": (150, 50),
    "2:1 (FP costly, industry-aligned)": (100, 50),
}

for label, (cfp, cfn) in cost_scenarios_to_test.items():
    best_thresh, best_cost = None, float("inf")
    for t in np.arange(0.1, 0.9, 0.01):
        preds = (probs_xgb > t).astype(int)
        fp = ((preds == 1) & (y_test == 0)).sum()
        fn = ((preds == 0) & (y_test == 1)).sum()
        total_cost = fp * cfp + fn * cfn
        if total_cost < best_cost:
            best_cost, best_thresh = total_cost, t
    print(f"[{label}] threshold={best_thresh:.2f}, cost=₹{best_cost:,}")
    print(classification_report(y_test, probs_xgb > best_thresh))


| Scenario | Threshold | Cost | Precision | Recall |
|---|---|---|---|---|
| 6:1 (FN costly, original) | 0.51 | ₹641,050 | 0.37 | 0.54 |
| 3:1 (FP costly, milder) | 0.89 | ₹143,100 | 0.92 | 0.25 |
| 2:1 (FP costly, industry-aligned) | 0.88 | ₹138,850 | 0.91 | 0.25 |

Two things stand out. First, the FP-costly scenarios push the threshold
sharply higher (0.88-0.89 vs. 0.51) and recall drops hard, from 0.54 to
0.25 — the model ends up catching only 1 in 4 actually-bad orders under
either FP-costly assumption. Second, the 3:1 and 2:1 results are nearly
identical despite the ratio changing, which means the marginal trade-off
in this region of the precision-recall curve is steep: past a certain
point, catching each additional true positive requires a disproportionate
jump in false positives, so milder FP-costly ratios don't meaningfully
change the outcome versus more aggressive ones. This is a real feature of
this model's precision-recall curve, visible as the steep drop after
roughly 0.5 recall in the curve plotted above — not a modeling artifact.

**This wasn't re-verified against the seller_bad_rate over-flagging finding
above** — that analysis was done entirely under the 6:1 scenario's
threshold (0.51). Whether the same false-positive concentration pattern
holds at threshold 0.88-0.89 is an open question this notebook doesn't
answer; the flag rate overall is much lower there (recall 0.25 vs 0.54),
so the shape of the false-positive distribution may well differ. Worth
flagging as a gap rather than assuming the earlier finding still applies.


## Final model

The model itself is unchanged: **AUC-ROC 0.751, AUC-PR 0.532** — these are
threshold-independent and hold regardless of which cost scenario is used.

What changed is that the app no longer commits to one threshold. It shows
all three scenarios side by side and defaults to the **2:1
industry-aligned scenario (threshold 0.88, precision 0.91, recall 0.25)**
for live scoring, while letting a reviewer switch to the original 6:1
view (threshold 0.51, precision 0.37, recall 0.54) or explore any point
in between. This reflects that the true cost ratio is a business decision
this project surfaces evidence for, rather than one this notebook can
settle unilaterally.

The training + artifact-export script lives in `src/train_final_model.py`
— running it end-to-end regenerates every file the Streamlit app depends
on, including `cost_scenarios.json`, which lets the app compute cost and
the optimal threshold for any of the three scenarios live, without
retraining.
